In [39]:
import geopandas as gpd
import pandas as pd
import numpy as np

print("1. Loading your REAL map shapes...")
# Load the actual map you used for your Streamlit app
geo_df = gpd.read_file('gemeenten.geojson')

# Extract whatever the municipality column is named (usually 'gemeentenaam')
# If your column is named differently, change it here!
region_col = 'gemeentenaam' 

print("2. Generating Test Health Scores...")
# Create a dataframe to hold our test scenarios
scenarios_df = pd.DataFrame({
    region_col: geo_df[region_col]
})

# Generate fake health risk scores (0-100) so we can test the HTML colors
# Baseline: Random score between 10 and 60
scenarios_df['risk_2025_baseline'] = np.random.randint(10, 60, size=len(geo_df))

# 2050 Baseline: Gets hotter, so scores go up (+20)
scenarios_df['risk_2050_baseline'] = scenarios_df['risk_2025_baseline'] + 20

# 2075 Baseline: Gets much hotter (+40)
scenarios_df['risk_2075_baseline'] = scenarios_df['risk_2025_baseline'] + 40

# 2050 Greening: Drops the 2050 risk by 15 points
scenarios_df['risk_2050_greening'] = scenarios_df['risk_2050_baseline'] - 15


print("3. Merging and Exporting...")
final_map = geo_df.merge(scenarios_df, on=region_col)

# THE FIX: Translate Dutch coordinates to standard GPS!
final_map = final_map.to_crs(epsg=4326)

# Overwrite the empty dummy file with this real, GPS-aligned map!
final_map.to_file('map_scenarios.geojson', driver='GeoJSON')
print("✅ Success! Hard refresh your HTML page.")

# Overwrite the empty dummy file with this real, colored map!
final_map.to_file('map_scenarios.geojson', driver='GeoJSON')
print("✅ Success! Refresh your HTML page.")

1. Loading your REAL map shapes...
2. Generating Test Health Scores...
3. Merging and Exporting...
✅ Success! Hard refresh your HTML page.
✅ Success! Refresh your HTML page.


In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np

print("1. Loading Official Neighborhood shapes (GPKG)...")
# 1. LOAD THE GEOPACKAGE
# We specify layer='buurten' to ensure we only grab the 14,000 neighborhoods, 
# not the municipalities or districts that might also be inside the file.
geo_df = gpd.read_file('wijkenbuurten_2024.gpkg', layer='buurten') 

# 2. CHECK THE MERGE COLUMN
# CBS usually names this 'buurtcode'. If you get a KeyError later, it might be capitalized as 'BUURTCODE'.
region_col = 'buurtcode' 

print("2. Generating Neighborhood Scenarios...")
scenarios_df = pd.DataFrame({region_col: geo_df[region_col]})

# Generate fake health risk scores (0-100) for all 14,000+ neighborhoods
scenarios_df['risk_2025_baseline'] = np.random.randint(10, 60, size=len(geo_df))
scenarios_df['risk_2050_baseline'] = scenarios_df['risk_2025_baseline'] + 20
scenarios_df['risk_2075_baseline'] = scenarios_df['risk_2025_baseline'] + 40
scenarios_df['risk_2050_greening'] = scenarios_df['risk_2050_baseline'] - 15

print("3. Compressing, Merging, and Exporting...")
final_map = geo_df.merge(scenarios_df, on=region_col)

# 3. CRITICAL SURVIVAL STEP: Heavy Compression!
# The original CBS file is in Dutch coordinates (Amersfoort/RD), which uses meters.
# A tolerance of 100 simplifies borders by removing zig-zags smaller than 100 meters.
final_map['geometry'] = final_map['geometry'].simplify(tolerance=100)

# 4. Translate to standard GPS and export for the web browser
final_map = final_map.to_crs(epsg=4326)
final_map.to_file('map_scenarios.geojson', driver='GeoJSON')

print("✅ Neighborhood Map Compressed and Exported!")

1. Loading Official Neighborhood shapes (GPKG)...
2. Generating Neighborhood Scenarios...
3. Compressing, Merging, and Exporting...
✅ Neighborhood Map Compressed and Exported!


In [16]:
import pandas as pd
import numpy as np

# 1. LOAD DATA with specific 'dot' handling for CBS suppression [cite: 86]
print("1. Loading and Slimming Pillars...")
health_df = pd.read_csv('health_data.csv', delimiter=';', na_values='.')
cbs_raw = pd.read_excel('kwb2024.xlsx', na_values='.')
climate_df = pd.read_csv('Heat_values_1.csv', na_values='.')

# 2. SLIM CBS DATA (Prevents Fragmentation Warning)
# Selecting target features: Population, Elderly, Low Income, and Rental % [cite: 12, 15, 18]
kwb_cols = ['gwb_code_10', 'recs', 'regio', 'a_inw', 'a_65_oo', 'p_ink_li', 'p_huurw']
cbs_bu = cbs_raw[cbs_raw['recs'].astype(str).str.contains('Buurt', case=False)][kwb_cols].copy() # [cite: 160]

# 3. FILTER HEALTH DATA
# Isolate Buurt level[cite: 160], specific age group (18+), and most recent period.
health_bu = health_df[
    (health_df['SoortRegio_2'].astype(str).str.contains('Buurt', case=False)) &
    (health_df['Leeftijd'].astype(str).str.contains('18 jaar of ouder', case=False))
].copy()

# 4. STANDARDIZE THE GOLDEN KEY (Keep 'BU' + 8 digits) [cite: 164]
def clean_key(series):
    return series.astype(str).str.strip().str.upper()

health_bu['MergeKey'] = clean_key(health_bu['Codering_3'])
cbs_bu['MergeKey'] = clean_key(cbs_bu['gwb_code_10'])
climate_df['MergeKey'] = clean_key(climate_df['buurtcode'])

# 5. THE NEIGHBORHOOD MERGE
master_df = pd.merge(health_bu, cbs_bu, on='MergeKey', how='inner')
master_df = pd.merge(master_df, climate_df, on='MergeKey', how='inner')

# 6. CALCULATE VULNERABILITY SCORE
# Risk = 100% minus those with 'Good Health'
target_in = 'GoedErvarenGezondheid_4'
target_out = 'Vulnerability_Score'

master_df[target_in] = pd.to_numeric(master_df[target_in], errors='coerce')
master_df[target_out] = 100 - master_df[target_in]

# 7. FINAL CLEANING & FEATURE ENGINEERING
# Remove neighborhoods where health data was suppressed (the 'dot' issue) [cite: 86]
final_master = master_df.dropna(subset=[target_out, 'PET_gem']).copy()

# Elderly % = (Residents 65+ / Total Inhabitants) * 100 [cite: 211, 198]
final_master['p_elderly'] = (pd.to_numeric(final_master['a_65_oo'], errors='coerce') / 
                             pd.to_numeric(final_master['a_inw'], errors='coerce')) * 100

# Select professional headers for the Ministry dashboard
output_cols = ['MergeKey', 'regio', target_out, 'PET_gem', 'p_ink_li', 'p_elderly', 'p_huurw']
final_master = final_master[output_cols].rename(columns={'MergeKey': 'buurtcode', 'regio': 'buurtnaam'})

print(f"✅ SUCCESS: {len(final_master)} neighborhoods found with full health & climate data.")
final_master.to_csv('Neighborhood_Master_Data.csv', index=False)

1. Loading and Slimming Pillars...
✅ SUCCESS: 0 neighborhoods found with full health & climate data.


In [42]:
import pandas as pd
import numpy as np

print("1. Loading Raw Data Pillars...")
# Load with proper handling of missing values from the start
health_df = pd.read_csv('health_data.csv', delimiter=';', na_values=['.', ''])
cbs_df = pd.read_excel('kwb2024.xlsx', na_values=['.', '']) 
climate_df = pd.read_csv('Heat_values_1.csv', na_values=['.', '']) 

print("\n2. Standardize Keys FIRST (before any filtering)...")
health_df['MergeKey'] = health_df['Codering_3'].astype(str).str.strip().str.upper()
cbs_df['MergeKey'] = cbs_df['gwb_code_10'].astype(str).str.strip().str.upper()
climate_df['MergeKey'] = climate_df['buurtcode'].astype(str).str.strip().str.upper()

print("\n3. Filter for Neighborhoods (BU only)...")
health_bu = health_df[health_df['MergeKey'].str.startswith('BU')].copy()
cbs_bu = cbs_df[cbs_df['MergeKey'].str.startswith('BU')].copy()
climate_bu = climate_df[climate_df['MergeKey'].str.startswith('BU')].copy()

print(f"   Health neighborhoods: {len(health_bu)}")
print(f"   CBS neighborhoods: {len(cbs_bu)}")
print(f"   Climate neighborhoods: {len(climate_bu)}")

print("\n4. Merge carefully with both sides...")
# Start with CBS (which has income data) and merge health INTO it
master_df = cbs_bu.copy()
print(f"   Starting with CBS: {len(master_df)} rows")

# Merge health data - use left merge from CBS so we keep all CBS data
master_df = pd.merge(master_df, health_bu, on='MergeKey', how='left', suffixes=('_cbs', '_health'))
print(f"   After adding health: {len(master_df)} rows")

# Merge climate data
master_df = pd.merge(master_df, climate_bu, on='MergeKey', how='left')
print(f"   After adding climate: {len(master_df)} rows")

print("\n5. Feature Engineering & Data Cleaning...")

# Clean data format issues (handle Dutch decimals and CBS suppression)
print("\n  Cleaning data format issues...")
for col in ['GoedErvarenGezondheid_4', 'p_ink_li', 'HeelVeelStressAfg4Weken_17']:
    if col in master_df.columns:
        # Replace '.' with empty string, then comma with period
        master_df[col] = master_df[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.')
        # Convert to numeric (will become NaN for empty strings)
        master_df[col] = pd.to_numeric(master_df[col], errors='coerce')

# Convert other numeric columns
for col in ['PET_gem', 'p_huurw', 'a_65_oo', 'a_inw', 'a_00_14', 'p_bj_me10']:
    if col in master_df.columns:
        master_df[col] = pd.to_numeric(master_df[col], errors='coerce')

# CALCULATE NEW FEATURES
# p_elderly (age 65+)
if 'a_65_oo' in master_df.columns and 'a_inw' in master_df.columns:
    master_df['p_elderly'] = (master_df['a_65_oo'] / master_df['a_inw']) * 100

# p_children (age 0-14) - NEW
if 'a_00_14' in master_df.columns and 'a_inw' in master_df.columns:
    master_df['p_children'] = (master_df['a_00_14'] / master_df['a_inw']) * 100

# p_bj_me10 (old buildings %) - already exists, just ensure it's numeric
# p_stress - already exists as HeelVeelStressAfg4Weken_17

# Handle duplicate neighborhoods (aggregate by taking mean)
print("  Removing duplicates...")
print(f"   Rows before dedup: {len(master_df)}")
final_master = master_df[[
    'MergeKey', 
    'GoedErvarenGezondheid_4', 
    'PET_gem', 
    'p_ink_li', 
    'p_elderly', 
    'p_huurw',
    'p_children',
    'p_bj_me10',
    'HeelVeelStressAfg4Weken_17'
]].copy()

# Group by neighborhood and take the mean (handles duplicate rows from health data periods)
final_master = final_master.groupby('MergeKey', as_index=False).mean(numeric_only=True)
print(f"   Rows after dedup: {len(final_master)}")

# Drop rows with missing critical values
final_master = final_master.dropna(subset=['GoedErvarenGezondheid_4'])

print(f"\n✅ FINAL DATASET: {len(final_master)} neighborhoods ready for modeling!")
print(f"\n  Data quality in final dataset:")
for col in ['GoedErvarenGezondheid_4', 'PET_gem', 'p_ink_li', 'p_elderly', 'p_huurw', 'p_children', 'p_bj_me10', 'HeelVeelStressAfg4Weken_17']:
    if col in final_master.columns:
        valid = final_master[col].notna().sum()
        pct = 100 * valid / len(final_master) if len(final_master) > 0 else 0
        mn = final_master[col].min()
        mx = final_master[col].max()
        print(f"    - {col}: {valid}/{len(final_master)} ({pct:.0f}%) | Range: {mn:.1f} to {mx:.1f}")

print(f"\nSample (first 5 rows):")
print(final_master.head())

print(f"\n💾 Saving to CSV...")
final_master.to_csv('Neighborhood_Master_Data.csv', index=False)


1. Loading Raw Data Pillars...

2. Standardize Keys FIRST (before any filtering)...

3. Filter for Neighborhoods (BU only)...
   Health neighborhoods: 43722
   CBS neighborhoods: 14574
   Climate neighborhoods: 14574

4. Merge carefully with both sides...
   Starting with CBS: 14574 rows
   After adding health: 43722 rows
   After adding climate: 43722 rows

5. Feature Engineering & Data Cleaning...

  Cleaning data format issues...


C:\Users\spygl\AppData\Local\Temp\ipykernel_23756\2253827754.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cbs_df['MergeKey'] = cbs_df['gwb_code_10'].astype(str).str.strip().str.upper()


  Removing duplicates...
   Rows before dedup: 43722
   Rows after dedup: 14574

✅ FINAL DATASET: 11734 neighborhoods ready for modeling!

  Data quality in final dataset:
    - GoedErvarenGezondheid_4: 11734/11734 (100%) | Range: 436.7 to 910.7
    - PET_gem: 11644/11734 (99%) | Range: 35.0 to 50.0
    - p_ink_li: 11678/11734 (100%) | Range: 7.1 to 95.1
    - p_elderly: 11734/11734 (100%) | Range: 0.0 to 100.0
    - p_huurw: 11689/11734 (100%) | Range: 0.0 to 100.0
    - p_children: 11734/11734 (100%) | Range: 0.0 to 39.4
    - p_bj_me10: 11689/11734 (100%) | Range: 0.0 to 100.0
    - HeelVeelStressAfg4Weken_17: 11734/11734 (100%) | Range: 76.3 to 459.0

Sample (first 5 rows):
     MergeKey  GoedErvarenGezondheid_4  PET_gem  p_ink_li  p_elderly  p_huurw  \
0  BU00140000               813.000000     44.0      63.8   6.954689     87.0   
1  BU00140001               809.000000     44.0      65.5   7.383513     88.0   
2  BU00140002               777.333333     45.0      61.9   8.750000  

In [ ]:
# 1. IMPORTS
from sklearn.linear_model import LinearRegression
import pandas as pd
import geopandas as gpd
import numpy as np

# 2. LOAD YOUR MASTER DATA
df = pd.read_csv('Neighborhood_Master_Data.csv')

# Remove any rows with missing values in features
df = df.dropna(subset=['GoedErvarenGezondheid_4', 'PET_gem', 'p_ink_li', 'p_elderly', 'p_children', 'p_bj_me10', 'HeelVeelStressAfg4Weken_17',])
print(f"Neighborhoods after removing NaNs: {len(df)}")

# 3. DEFINE THE MODEL (NOW WITH 7 FEATURES)
# Target: GoedErvarenGezondheid_4 (Good Health Perception - INVERSE of vulnerability)
# Higher values = better health = LOWER vulnerability score needed
# 
# Features (7 total):
# Original 4:
#   - PET_gem (Perceived Temperature - Heat exposure)
#   - p_ink_li (Low Income %)
#   - p_elderly (Age 65+ %)
# New 3:
#   - p_children (Age 0-14 %) - Vulnerable age group
#   - p_bj_me10 (Old Buildings % built before 2010) - Poor insulation/cooling
#   - HeelVeelStressAfg4Weken_17 (High Stress %) - Mental health vulnerability

X = df[['PET_gem', 'p_ink_li', 'p_elderly', 'p_children', 'p_bj_me10', 'HeelVeelStressAfg4Weken_17',]]
y = 100 - df['GoedErvarenGezondheid_4']  # Convert: invert so higher = worse health outcome

# Train the prediction model
model = LinearRegression()
model.fit(X, y)

print("✅ Model Trained Successfully on " + str(len(df)) + " neighborhoods!")
print(f"\n🚨 Model Coefficients (Impact on Health Vulnerability):")
print(f"Intercept (Baseline Risk): {model.intercept_:.4f}")
for feature, coef in zip(X.columns, model.coef_):
    direction = "increases" if coef > 0 else "decreases"
    abs_coef = abs(coef)
    print(f" - {feature}: {coef:+.6f} ({direction} risk) | Impact: {abs_coef:.2f}")

# Calculate R-squared for model quality check
from sklearn.metrics import r2_score
y_pred = model.predict(X)
r2 = r2_score(y, y_pred)
print(f"\n📊 Model Quality (R²): {r2:.4f}")

# 4. PRE-GENERATE SCENARIOS
# Baseline 2025
df['risk_2025_baseline'] = model.predict(X)

# 2050 Scenario: Increase PET temperature by 2 degrees
X_2050 = X.copy()
X_2050['PET_gem'] += 2.0
df['risk_2050_baseline'] = model.predict(X_2050)

# 2075 Scenario: Increase PET temperature by 4 degrees
X_2075 = X.copy()
X_2075['PET_gem'] += 4.0
df['risk_2075_baseline'] = model.predict(X_2075)

# Policy Simulation: Multiple interventions
# - Reduce rental housing dependency (new housing/ownership programs)
# - Reduce stress through community programs (stress drops 15%)
X_policy = X_2050.copy()
X_policy['HeelVeelStressAfg4Weken_17'] -= 15  
X_policy['HeelVeelStressAfg4Weken_17'] = X_policy['HeelVeelStressAfg4Weken_17'].clip(lower=0)
df['risk_2050_greening'] = model.predict(X_policy)

print(f"\n📊 Raw Risk Score Statistics (before normalization):")
for col in ['risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening']:
    mn = df[col].min()
    mx = df[col].max()
    avg = df[col].mean()
    print(f"   {col}: Min={mn:.1f}, Max={mx:.1f}, Avg={avg:.1f}")

# NORMALIZE TO 0-100 SCALE FOR VISUALIZATION
# Find global min/max across all scenarios for consistent scaling
all_risks = pd.concat([
    df['risk_2025_baseline'], 
    df['risk_2050_baseline'], 
    df['risk_2075_baseline'],
    df['risk_2050_greening']
])
global_min = all_risks.min()
global_max = all_risks.max()

print(f"\n📏 Normalization range: {global_min:.1f} to {global_max:.1f}")

# Normalize all risk columns to 0-100 scale
for col in ['risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening']:
    df[col] = 100 * (df[col] - global_min) / (global_max - global_min)

print(f"\n✅ Normalized Risk Score Statistics (0-100 scale):")
for col in ['risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening']:
    mn = df[col].min()
    mx = df[col].max()
    avg = df[col].mean()
    print(f"   {col}: Min={mn:.1f}, Max={mx:.1f}, Avg={avg:.1f}")

# 5. CONNECT TO THE NEIGHBORHOOD MAP
geo_df = gpd.read_file('wijkenbuurten_2024.gpkg', layer='buurten')
print(f"\n🗺️ Loaded {len(geo_df)} neighborhoods from GeoPackage")

# Key step: Standardize the buurtcode in the geopackage to match our MergeKey
geo_df['buurtcode_clean'] = geo_df['buurtcode'].astype(str).str.strip().str.upper()
df['MergeKey_clean'] = df['MergeKey'].astype(str).str.strip().str.upper()

# Merge predictions into map
print(f"   Merging predictions...")
final_map = geo_df.merge(df[['MergeKey_clean', 'risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening']], 
                          left_on='buurtcode_clean', right_on='MergeKey_clean', how='left')

# Check merge success
merged_count = final_map['risk_2025_baseline'].notna().sum()
print(f"   ✅ Successfully merged {merged_count} neighborhoods with predictions")

# 6. EXPORT FOR WEB - with optimizations
final_map = final_map.to_crs(epsg=4326)
final_map['geometry'] = final_map['geometry'].simplify(tolerance=0.001)

# Select only essential columns for the GeoJSON (smaller file size)
export_cols = ['buurtcode', 'risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening', 'geometry']
final_map_export = final_map[[c for c in export_cols if c in final_map.columns]].copy()

final_map_export.to_file('map_scenarios.geojson', driver='GeoJSON')

print(f"\n✅ GeoJSON exported: {merged_count} neighborhoods with normalized risk predictions (0-100 scale)")
print("   Ready for web dashboard!")


Neighborhoods after removing NaNs: 11554
✅ Model Trained Successfully on 11554 neighborhoods!

🚨 Model Coefficients (Impact on Health Vulnerability):
Intercept (Baseline Risk): -1332.3467
 - PET_gem: +4.324101 (increases risk) | Impact: 4.32
 - p_ink_li: +3.402009 (increases risk) | Impact: 3.40
 - p_elderly: +4.047031 (increases risk) | Impact: 4.05
 - p_children: +2.990138 (increases risk) | Impact: 2.99
 - p_bj_me10: +0.608858 (increases risk) | Impact: 0.61
 - HeelVeelStressAfg4Weken_17: +0.930002 (increases risk) | Impact: 0.93

📊 Model Quality (R²): 0.7118

📊 Raw Risk Score Statistics (before normalization):
   risk_2025_baseline: Min=-793.8, Max=-379.9, Avg=-645.7
   risk_2050_baseline: Min=-785.1, Max=-371.2, Avg=-637.0
   risk_2075_baseline: Min=-776.5, Max=-362.6, Avg=-628.4
   risk_2050_greening: Min=-799.1, Max=-385.2, Avg=-651.0

📏 Normalization range: -799.1 to -362.6

✅ Normalized Risk Score Statistics (0-100 scale):
   risk_2025_baseline: Min=1.2, Max=96.0, Avg=35.1
   

In [36]:


# === MODEL SUMMARY FOR YOUR CAPSTONE THESIS ===
print("\n" + "="*70)
print("HEAT VULNERABILITY MODEL - NEIGHBORHOOD ANALYSIS")
print("="*70)

print("\n# Define Features (X) and Target (y)")
print("X = final_master[['PET_gem', 'p_ink_li', 'p_elderly', 'p_huurw']]")
print("y = 100 - final_master['GoedErvarenGezondheid_4']  # (Inverted: lower good health = higher vulnerability)")

print("\n# Train the model")
print("model = LinearRegression()")
print("model.fit(X, y)")

print("\n🚨 Model Trained on Real Neighborhood Data")
print(f"Intercept (Baseline Risk): {model.intercept_:.4f}")
for feature, coef in zip(X.columns, model.coef_):
    print(f" - {feature}: {coef:+.4f}")

print("\n" + "="*70)
print("INTERPRETATION:")
print("="*70)
print("""
Each coefficient shows how much vulnerability INCREASES for every 1-unit increase:

✓ PET_gem (+4.46): Every +1°C perceived temperature = +4.46% vulnerability
✓ p_ink_li (+3.25): Every +1% low-income residents = +3.25% vulnerability  
✓ p_elderly (+1.43): Every +1% elderly residents = +1.43% vulnerability
✓ p_huurw (+1.40): Every +1% rental housing = +1.40% vulnerability

This validates that heat vulnerability is driven by BOTH climate AND socio-economic factors.
""")


HEAT VULNERABILITY MODEL - NEIGHBORHOOD ANALYSIS

# Define Features (X) and Target (y)
X = final_master[['PET_gem', 'p_ink_li', 'p_elderly', 'p_huurw']]
y = 100 - final_master['GoedErvarenGezondheid_4']  # (Inverted: lower good health = higher vulnerability)

# Train the model
model = LinearRegression()
model.fit(X, y)

🚨 Model Trained on Real Neighborhood Data
Intercept (Baseline Risk): -1047.8956
 - PET_gem: +4.4612
 - p_ink_li: +3.2462
 - p_elderly: +1.4273
 - p_huurw: +1.4013

INTERPRETATION:

Each coefficient shows how much vulnerability INCREASES for every 1-unit increase:

✓ PET_gem (+4.46): Every +1°C perceived temperature = +4.46% vulnerability
✓ p_ink_li (+3.25): Every +1% low-income residents = +3.25% vulnerability  
✓ p_elderly (+1.43): Every +1% elderly residents = +1.43% vulnerability
✓ p_huurw (+1.40): Every +1% rental housing = +1.40% vulnerability

This validates that heat vulnerability is driven by BOTH climate AND socio-economic factors.



In [1]:
# 1. IMPORTS
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pandas as pd
import geopandas as gpd
import numpy as np

# 2. LOAD YOUR MASTER DATA
df = pd.read_csv('Neighborhood_Master_Data.csv')

# Remove any rows with missing values in features
df = df.dropna(subset=['GoedErvarenGezondheid_4', 'PET_gem', 'p_ink_li', 'p_elderly', 'p_children', 'p_bj_me10', 'HeelVeelStressAfg4Weken_17'])
print(f"Neighborhoods after removing NaNs: {len(df)}")

# 3. DEFINE THE MODEL 
# Target: GoedErvarenGezondheid_4 (Good Health Perception - INVERSE of vulnerability)
# Higher values = better health = LOWER vulnerability score needed
# 
# Features:
# Original:
#   - PET_gem (Perceived Temperature - Heat exposure)
#   - p_ink_li (Low Income %)
#   - p_elderly (Age 65+ %)
# New:
#   - p_children (Age 0-14 %) - Vulnerable age group
#   - p_bj_me10 (Old Buildings % built before 2010) - Poor insulation/cooling
#   - HeelVeelStressAfg4Weken_17 (High Stress %) - Mental health vulnerability

X = df[['PET_gem', 'p_ink_li', 'p_elderly', 'p_children', 'p_bj_me10', 'HeelVeelStressAfg4Weken_17']]
y = 100 - df['GoedErvarenGezondheid_4']  # Convert: invert so higher = worse health outcome

# Train the prediction model
model = LinearRegression()
model.fit(X, y)

print(f"✅ Model Trained Successfully on {len(df)} neighborhoods!")
print(f"\n🚨 Model Coefficients (Impact on Health Vulnerability):")
print(f"Intercept (Baseline Risk): {model.intercept_:.4f}")
for feature, coef in zip(X.columns, model.coef_):
    direction = "increases" if coef > 0 else "decreases"
    abs_coef = abs(coef)
    print(f" - {feature}: {coef:+.6f} ({direction} risk) | Impact: {abs_coef:.2f}")

# --- MODEL ACCURACY METRICS ---
y_pred = model.predict(X)
r2 = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"\n📊 Model Quality Metrics:")
print(f" - R-squared (R²): {r2:.4f} (Closer to 1.0 is better)")
print(f" - Mean Absolute Error (MAE): {mae:.4f} (Average error in risk score points)")
print(f" - Root Mean Squared Error (RMSE): {rmse:.4f} (Penalizes larger errors)")
# ------------------------------

# 4. PRE-GENERATE SCENARIOS
# Baseline 2025
df['risk_2025_baseline'] = model.predict(X)

# 2050 Scenario: Increase PET temperature by 2 degrees
X_2050 = X.copy()
X_2050['PET_gem'] += 2.0
df['risk_2050_baseline'] = model.predict(X_2050)

# 2075 Scenario: Increase PET temperature by 4 degrees
X_2075 = X.copy()
X_2075['PET_gem'] += 4.0
df['risk_2075_baseline'] = model.predict(X_2075)

# Policy Simulation: Multiple interventions
# - Reduce rental housing dependency (new housing/ownership programs)
# - Reduce stress through community programs (stress drops 15%)
X_policy = X_2050.copy()
X_policy['HeelVeelStressAfg4Weken_17'] -= 15  
X_policy['HeelVeelStressAfg4Weken_17'] = X_policy['HeelVeelStressAfg4Weken_17'].clip(lower=0)
df['risk_2050_greening'] = model.predict(X_policy)

print(f"\n📊 Raw Risk Score Statistics (un-normalized):")
for col in ['risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening']:
    mn = df[col].min()
    mx = df[col].max()
    avg = df[col].mean()
    print(f"   {col}: Min={mn:.1f}, Max={mx:.1f}, Avg={avg:.1f}")

# (Normalization code removed from here!)

# 5. CONNECT TO THE NEIGHBORHOOD MAP
geo_df = gpd.read_file('wijkenbuurten_2024.gpkg', layer='buurten')
print(f"\n🗺️ Loaded {len(geo_df)} neighborhoods from GeoPackage")

# Key step: Standardize the buurtcode in the geopackage to match our MergeKey
geo_df['buurtcode_clean'] = geo_df['buurtcode'].astype(str).str.strip().str.upper()
df['MergeKey_clean'] = df['MergeKey'].astype(str).str.strip().str.upper()

# Merge predictions into map
print(f"   Merging predictions...")
final_map = geo_df.merge(df[['MergeKey_clean', 'risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening']], 
                          left_on='buurtcode_clean', right_on='MergeKey_clean', how='left')

# Check merge success
merged_count = final_map['risk_2025_baseline'].notna().sum()
print(f"   ✅ Successfully merged {merged_count} neighborhoods with predictions")

# 6. EXPORT FOR WEB - with optimizations
final_map = final_map.to_crs(epsg=4326)
final_map['geometry'] = final_map['geometry'].simplify(tolerance=0.001)

# Select only essential columns for the GeoJSON (smaller file size)
export_cols = ['buurtcode', 'risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening', 'geometry']
final_map_export = final_map[[c for c in export_cols if c in final_map.columns]].copy()

final_map_export.to_file('map_scenarios.geojson', driver='GeoJSON')

print(f"\n✅ GeoJSON exported: {merged_count} neighborhoods with RAW risk predictions")
print("   Ready for web dashboard!")

C:\Users\spygl\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Neighborhoods after removing NaNs: 11554
✅ Model Trained Successfully on 11554 neighborhoods!

🚨 Model Coefficients (Impact on Health Vulnerability):
Intercept (Baseline Risk): -1332.3467
 - PET_gem: +4.324101 (increases risk) | Impact: 4.32
 - p_ink_li: +3.402009 (increases risk) | Impact: 3.40
 - p_elderly: +4.047031 (increases risk) | Impact: 4.05
 - p_children: +2.990138 (increases risk) | Impact: 2.99
 - p_bj_me10: +0.608858 (increases risk) | Impact: 0.61
 - HeelVeelStressAfg4Weken_17: +0.930002 (increases risk) | Impact: 0.93

📊 Model Quality Metrics:
 - R-squared (R²): 0.7118 (Closer to 1.0 is better)
 - Mean Absolute Error (MAE): 25.2740 (Average error in risk score points)
 - Root Mean Squared Error (RMSE): 34.4978 (Penalizes larger errors)

📊 Raw Risk Score Statistics (un-normalized):
   risk_2025_baseline: Min=-793.8, Max=-379.9, Avg=-645.7
   risk_2050_baseline: Min=-785.1, Max=-371.2, Avg=-637.0
   risk_2075_baseline: Min=-776.5, Max=-362.6, Avg=-628.4
   risk_2050_greeni

In [1]:
# 1. IMPORTS
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pandas as pd
import geopandas as gpd
import numpy as np

# 2. LOAD YOUR MASTER DATA
df = pd.read_csv('Neighborhood_Master_Data.csv')

# Remove any rows with missing values in features
df = df.dropna(subset=['GoedErvarenGezondheid_4', 'PET_gem', 'p_ink_li', 'p_elderly', 'p_children', 'p_bj_me10', 'HeelVeelStressAfg4Weken_17'])
print(f"Neighborhoods after removing NaNs: {len(df)}")

# 3. DEFINE THE MODEL 
X = df[['PET_gem', 'p_ink_li', 'p_elderly', 'p_children', 'p_bj_me10', 'HeelVeelStressAfg4Weken_17']]
y = 100 - df['GoedErvarenGezondheid_4']

model = LinearRegression()
model.fit(X, y)

print(f"✅ Model Trained Successfully on {len(df)} neighborhoods!")
print(f"\n🚨 Model Coefficients (Impact on Health Vulnerability):")
print(f"Intercept (Baseline Risk): {model.intercept_:.4f}")
for feature, coef in zip(X.columns, model.coef_):
    direction = "increases" if coef > 0 else "decreases"
    abs_coef = abs(coef)
    print(f" - {feature}: {coef:+.6f} ({direction} risk) | Impact: {abs_coef:.2f}")

y_pred = model.predict(X)
r2 = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"\n📊 Model Quality Metrics:")
print(f" - R-squared (R²): {r2:.4f}")
print(f" - Mean Absolute Error (MAE): {mae:.4f}")
print(f" - Root Mean Squared Error (RMSE): {rmse:.4f}")

# 4. PRE-GENERATE SCENARIOS
df['risk_2025_baseline'] = model.predict(X)

X_2050 = X.copy()
X_2050['PET_gem'] += 2.0
df['risk_2050_baseline'] = model.predict(X_2050)

X_2075 = X.copy()
X_2075['PET_gem'] += 4.0
df['risk_2075_baseline'] = model.predict(X_2075)

X_policy = X_2050.copy()
X_policy['HeelVeelStressAfg4Weken_17'] -= 15  
X_policy['HeelVeelStressAfg4Weken_17'] = X_policy['HeelVeelStressAfg4Weken_17'].clip(lower=0)
df['risk_2050_greening'] = model.predict(X_policy)

# 5. COMPUTE THEMATIC SUB-INDICES
# Each theme score is a weighted sum of its constituent variables,
# kept in raw units so JS can normalize them at render time.

# Theme 1: Heat Exposure — just PET_gem directly
df['theme_heat'] = df['PET_gem']

# Theme 2: Vulnerable Populations — elderly + children (equal weight)
df['theme_populations'] = df['p_elderly'] + df['p_children']

# Theme 3: Socioeconomic Risk — low income share
df['theme_socioeconomic'] = df['p_ink_li']

# Theme 4: Mental Health & Stress
df['theme_mental_health'] = df['HeelVeelStressAfg4Weken_17']

# Theme 5: Built Environment — pre-2010 building stock
df['theme_built_env'] = df['p_bj_me10']

print(f"\n📊 Theme Statistics:")
for col in ['theme_heat', 'theme_populations', 'theme_socioeconomic', 'theme_mental_health', 'theme_built_env']:
    mn = df[col].min()
    mx = df[col].max()
    avg = df[col].mean()
    print(f"   {col}: Min={mn:.2f}, Max={mx:.2f}, Avg={avg:.2f}")

# 6. CONNECT TO THE NEIGHBORHOOD MAP
geo_df = gpd.read_file('wijkenbuurten_2024.gpkg', layer='buurten')
print(f"\n🗺️ Loaded {len(geo_df)} neighborhoods from GeoPackage")

geo_df['buurtcode_clean'] = geo_df['buurtcode'].astype(str).str.strip().str.upper()
df['MergeKey_clean'] = df['MergeKey'].astype(str).str.strip().str.upper()

merge_cols = [
    'MergeKey_clean',
    # Composite scenario scores
    'risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening',
    # Thematic raw variable scores
    'theme_heat', 'theme_populations', 'theme_socioeconomic',
    'theme_mental_health', 'theme_built_env'
]

print(f"   Merging predictions...")
final_map = geo_df.merge(
    df[merge_cols],
    left_on='buurtcode_clean',
    right_on='MergeKey_clean',
    how='left'
)

merged_count = final_map['risk_2025_baseline'].notna().sum()
print(f"   ✅ Successfully merged {merged_count} neighborhoods with predictions")

# 7. EXPORT FOR WEB
final_map = final_map.to_crs(epsg=4326)
final_map['geometry'] = final_map['geometry'].simplify(tolerance=0.001)

export_cols = [
    'buurtcode', 'buurtnaam',
    # Scenario scores
    'risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening',
    # Theme scores
    'theme_heat', 'theme_populations', 'theme_socioeconomic',
    'theme_mental_health', 'theme_built_env',
    'geometry'
]
final_map_export = final_map[[c for c in export_cols if c in final_map.columns]].copy()

final_map_export.to_file('map_scenarios.geojson', driver='GeoJSON')

print(f"\n✅ GeoJSON exported: {merged_count} neighborhoods")
print("   Includes composite risk scores + 5 thematic variable layers")
print("   Ready for web dashboard!")


C:\Users\spygl\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Neighborhoods after removing NaNs: 11554
✅ Model Trained Successfully on 11554 neighborhoods!

🚨 Model Coefficients (Impact on Health Vulnerability):
Intercept (Baseline Risk): -1332.3467
 - PET_gem: +4.324101 (increases risk) | Impact: 4.32
 - p_ink_li: +3.402009 (increases risk) | Impact: 3.40
 - p_elderly: +4.047031 (increases risk) | Impact: 4.05
 - p_children: +2.990138 (increases risk) | Impact: 2.99
 - p_bj_me10: +0.608858 (increases risk) | Impact: 0.61
 - HeelVeelStressAfg4Weken_17: +0.930002 (increases risk) | Impact: 0.93

📊 Model Quality Metrics:
 - R-squared (R²): 0.7118
 - Mean Absolute Error (MAE): 25.2740
 - Root Mean Squared Error (RMSE): 34.4978

📊 Theme Statistics:
   theme_heat: Min=35.00, Max=50.00, Avg=44.34
   theme_populations: Min=0.00, Max=100.00, Avg=36.41
   theme_socioeconomic: Min=7.10, Max=95.10, Avg=39.23
   theme_mental_health: Min=76.33, Max=459.00, Avg=186.95
   theme_built_env: Min=0.00, Max=100.00, Avg=91.47

🗺️ Loaded 14668 neighborhoods from GeoPa

In [7]:
# ============================================================
# HEAT VULNERABILITY MODEL — Full Thematic Edition
# 6 themes drawn from: Neighborhood_Master_Data.csv,
#   health_data.csv, Heat_values_1.csv,
#   wijkenbuurten_2024.gpkg (CBS distances & socioeconomic)
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pandas as pd
import geopandas as gpd
import numpy as np


# ── HELPERS ─────────────────────────────────────────────────

def pct_rank(series, invert=False):
    """0-100 percentile rank. invert=True for 'safer = higher raw value'."""
    r = series.rank(pct=True, na_option='keep') * 100
    return (100 - r) if invert else r

def theme_score(df, col_defs):
    """Average pct-rank of available columns. col_defs = [(col, invert), ...]"""
    parts, used = [], []
    for (col, invert) in col_defs:
        if col in df.columns and df[col].notna().sum() > 100:
            parts.append(pct_rank(df[col], invert))
            used.append(("~" if invert else "+") + col)
    if not parts:
        return pd.Series([np.nan] * len(df), index=df.index)
    print(f"      {used}")
    return pd.concat(parts, axis=1).mean(axis=1)


# ============================================================
# 2. LOAD BASE MASTER DATA
# ============================================================
df = pd.read_csv('Neighborhood_Master_Data.csv')
df = df.dropna(subset=[
    'GoedErvarenGezondheid_4', 'PET_gem', 'p_ink_li',
    'p_elderly', 'p_children', 'p_bj_me10', 'HeelVeelStressAfg4Weken_17'
])
df['_mk'] = df['MergeKey'].astype(str).str.strip().str.upper()
print(f"Base dataset: {len(df)} neighborhoods")


# ============================================================
# 3. MERGE EXTRA HEALTH INDICATORS  (health_data.csv)
#    loneliness · anxiety · frailty · cooling access
# ============================================================
try:
    WANT = [
        'Codering_3',
        'Eenzaam_25',                            # Lonely %
        'SterkEenzaam_26',                       # Severely lonely %
        'EmotioneelEenzaam_27',                  # Emotionally lonely %
        'SociaalEenzaam_28',                     # Socially lonely %
        'HoogRisicoAngststoornisOfDepressie_14', # Anxiety/depression risk %
        'AngstDepressiegevoelensAfg4Weken_16',   # Anxiety/depression feelings %
        'BrozeGezondheid_18',                    # Frail health %
        'BrozeGezondheidPsychologischDomein_19', # Psychological frailty %
        'BrozeGezondheidFysiekDomein_20',        # Physical frailty %
        'KanGoedVerkoelingVindenBinnen_59',      # Can find indoor cooling %
        'KanGoedVerkoelingVindenBuiten_60',      # Can find outdoor cooling %
    ]
    
    # Optimized: Only load the specific columns we need into memory
    health_raw = pd.read_csv('health_data.csv', sep=';', usecols=lambda c: c in WANT)
    
    avail = [c for c in WANT if c in health_raw.columns]
    h = health_raw[avail].copy()
    for c in avail:
        if c != 'Codering_3':
            h[c] = pd.to_numeric(h[c], errors='coerce')
    h_agg = h.groupby('Codering_3', as_index=False).mean(numeric_only=True)
    h_agg['Codering_3'] = h_agg['Codering_3'].astype(str).str.strip().str.upper()
    df = df.merge(h_agg, left_on='_mk', right_on='Codering_3', how='left')
    df = df.drop(columns=['Codering_3'], errors='ignore')
    print(f"  health_data.csv: {len([c for c in WANT[1:] if c in df.columns])} indicators merged")
except Exception as e:
    print(f"  health_data.csv skipped: {e}")


# ============================================================
# 4. MERGE HEAT CLASSIFICATION  (Heat_values_1.csv)
#    RIVMClass: 1=mild  2=moderate  3=severe  4=extreme
# ============================================================
try:
    hc = pd.read_csv('Heat_values_1.csv')
    hc['_hk'] = hc['buurtcode'].astype(str).str.strip().str.upper()
    df = df.merge(hc[['_hk', 'RIVMClass']], left_on='_mk', right_on='_hk', how='left')
    df = df.drop(columns=['_hk'], errors='ignore')
    print(f"  Heat_values_1.csv: RIVMClass merged for {df['RIVMClass'].notna().sum()} neighborhoods")
except Exception as e:
    print(f"  Heat_values_1.csv skipped: {e}")


# ============================================================
# 5. DEFINE + TRAIN COMPOSITE 6-FACTOR MODEL
# ============================================================
X_COLS = ['PET_gem', 'p_ink_li', 'p_elderly', 'p_children',
          'p_bj_me10', 'HeelVeelStressAfg4Weken_17']
X = df[X_COLS]
y = 100 - df['GoedErvarenGezondheid_4']   # Invert: higher = worse health

# Rigorous Accuracy Check (Train-Test Split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
test_model = LinearRegression()
test_model.fit(X_train, y_train)
y_test_pred = test_model.predict(X_test)

print(f"\n--- True Model Accuracy (Test Set) ---")
print(f"R²:  {r2_score(y_test, y_test_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_test_pred):.4f} points off")

# Final Model for the Map (Learns from everything)
model = LinearRegression()
model.fit(X, y)

print("\nFinal Model Coefficients:")
for feat, coef in zip(X_COLS, model.coef_):
    print(f"  {feat:45s} {coef:+.4f}")


# ============================================================
# 6. PRE-GENERATE COMPOSITE SCENARIOS
#    All four scenarios normalised on a SHARED 0-100 scale
#    so cross-scenario comparisons remain valid.
# ============================================================
df['_r2025'] = model.predict(X)

X50 = X.copy(); X50['PET_gem'] += 2.0
df['_r2050'] = model.predict(X50)

X75 = X.copy(); X75['PET_gem'] += 4.0
df['_r2075'] = model.predict(X75)

Xpol = X50.copy()
Xpol['HeelVeelStressAfg4Weken_17'] = (Xpol['HeelVeelStressAfg4Weken_17'] - 15).clip(lower=0)
df['_rpol'] = model.predict(Xpol)

# Shared normalisation range
raw_all = pd.concat([df['_r2025'], df['_r2050'], df['_r2075'], df['_rpol']]).dropna()
r_min, r_max = raw_all.min(), raw_all.max()
for raw, out in [('_r2025','risk_2025_baseline'), ('_r2050','risk_2050_baseline'),
                 ('_r2075','risk_2075_baseline'), ('_rpol', 'risk_2050_greening')]:
    df[out] = ((df[raw] - r_min) / (r_max - r_min) * 100).round(1).clip(0, 100)
df = df.drop(columns=['_r2025','_r2050','_r2075','_rpol'])

print("\nScenario averages (shared 0-100 scale):")
for c in ['risk_2025_baseline','risk_2050_baseline','risk_2075_baseline','risk_2050_greening']:
    print(f"  {c}: mean={df[c].mean():.1f}  max={df[c].max():.1f}")


# ============================================================
# 7. PULL EXTRA CBS VARIABLES FROM GEOPACKAGE
#    (distances, green space, energy, housing, socioeconomic)
# ============================================================
geo_df = gpd.read_file('wijkenbuurten_2024.gpkg', layer='buurten')
geo_df['_bk'] = geo_df['buurtcode'].astype(str).str.strip().str.upper()
print(f"\nGeoPackage loaded: {len(geo_df)} neighborhoods")

CBS_MAP = {
    # Healthcare access
    'huisartsenpraktijk_gemiddelde_afstand_in_km':         'dist_gp',
    'ziekenhuis_incl_buitenpolikliniek_gem_afst_in_km':    'dist_hospital',
    'apotheek_gemiddelde_afstand_in_km':                   'dist_pharmacy',
    # Green / cooling
    'afstand_tot_park_of_plantsoen':                       'dist_park',
    'afstand_tot_openbaar_groen_totaal':                   'dist_green',
    'afstand_tot_bos':                                     'dist_forest',
    # Housing / energy
    'gemiddeld_gasverbruik_totaal':                        'gas_use',
    'gemiddeld_elektriciteitsverbruik_totaal':             'elec_use',
    'percentage_bouwjaarklasse_tot_2000':                  'pct_pre2000',
    # Socioeconomic
    'opleidingsniveau_laag':                               'pct_low_edu',
    'percentage_huishoudens_onder_of_rond_sociaal_minimum':'pct_social_min',
    # Social
    'percentage_eenpersoonshuishoudens':                   'pct_single_hh',
    'bevolkingsdichtheid_inwoners_per_km2':                'pop_density',
}
avail_cbs = {o: s for o, s in CBS_MAP.items() if o in geo_df.columns}
cbs_df = (geo_df[['_bk'] + list(avail_cbs.keys())]
          .rename(columns=avail_cbs)
          .groupby('_bk', as_index=False).first())

df = df.merge(cbs_df, left_on='_mk', right_on='_bk', how='left')
df = df.drop(columns=['_mk', '_bk'], errors='ignore')

# Fix: Clean CBS null flags (-99999999) to prevent data skew
for col in avail_cbs.values():
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] < -900, col] = np.nan

print(f"  CBS variables added and cleaned: {list(avail_cbs.values())}")


# ============================================================
# 8. COMPUTE 6 THEMATIC SCORES  (0-100 percentile rank)
#    Higher score = higher vulnerability on that dimension
#    ~ prefix = inverted (higher raw value = safer)
# ============================================================
THEME_COLS = [
    'theme_heat', 'theme_populations', 'theme_socioeconomic',
    'theme_mental_health', 'theme_built_env', 'theme_access'
]

print("\nComputing themes:")

# ── THEME 1: HEAT EXPOSURE ────────────────────────────────
print("  [1] Heat Exposure")
df['theme_heat'] = theme_score(df, [
    ('PET_gem',   False),
    ('RIVMClass', False),
])

# ── THEME 2: VULNERABLE POPULATIONS ──────────────────────
print("  [2] Vulnerable Populations")
df['theme_populations'] = theme_score(df, [
    ('p_elderly',                      False),
    ('p_children',                     False),
    ('pct_single_hh',                  False),
    ('BrozeGezondheid_18',             False),
    ('BrozeGezondheidFysiekDomein_20', False),
])

# ── THEME 3: SOCIOECONOMIC DEPRIVATION ───────────────────
print("  [3] Socioeconomic Deprivation")
df['theme_socioeconomic'] = theme_score(df, [
    ('p_ink_li',          False),
    ('pct_low_edu',       False),
    ('pct_social_min',    False),
    ('p_huurw',           False),
])

# ── THEME 4: MENTAL HEALTH & SOCIAL ISOLATION ────────────
print("  [4] Mental Health & Social Isolation")
df['theme_mental_health'] = theme_score(df, [
    ('HeelVeelStressAfg4Weken_17',           False),
    ('Eenzaam_25',                            False),
    ('SterkEenzaam_26',                       False),
    ('HoogRisicoAngststoornisOfDepressie_14', False),
    ('AngstDepressiegevoelensAfg4Weken_16',   False),
    ('BrozeGezondheidPsychologischDomein_19', False),
])

# ── THEME 5: BUILT ENVIRONMENT ───────────────────────────
print("  [5] Built Environment")
df['theme_built_env'] = theme_score(df, [
    ('p_bj_me10',                      False),
    ('pct_pre2000',                    False),
    ('gas_use',                        False),
    ('KanGoedVerkoelingVindenBinnen_59', True),  # INVERT: higher = safer
])

# ── THEME 6: ACCESS TO CARE & GREEN/COOLING SPACE ────────
print("  [6] Access & Green Space")
df['theme_access'] = theme_score(df, [
    ('dist_gp',                        False),
    ('dist_hospital',                  False),
    ('dist_pharmacy',                  False),
    ('dist_park',                      False),
    ('dist_green',                     False),
    ('KanGoedVerkoelingVindenBuiten_60', True),  # INVERT: higher = safer
])

print("\nTheme statistics (0-100):")
for col in THEME_COLS:
    s = df[col]
    print(f"  {col:30s}  n={s.notna().sum():5d}  "
          f"mean={s.mean():.1f}  p25={s.quantile(.25):.1f}  "
          f"p75={s.quantile(.75):.1f}")


# ============================================================
# 9. MERGE WITH GEO + EXPORT GeoJSON
# ============================================================
SCEN_COLS  = ['risk_2025_baseline', 'risk_2050_baseline',
              'risk_2075_baseline', 'risk_2050_greening']
MERGE_COLS = ['MergeKey'] + SCEN_COLS + THEME_COLS

df['_mk2'] = df['MergeKey'].astype(str).str.strip().str.upper()
final_map = geo_df.merge(df[MERGE_COLS + ['_mk2']],
                          left_on='_bk', right_on='_mk2', how='left')
final_map = final_map.drop(columns=['_bk', '_mk2', 'MergeKey'], errors='ignore')

merged_count = final_map['risk_2025_baseline'].notna().sum()
print(f"\nMerged {merged_count} / {len(geo_df)} neighborhoods with predictions")

final_map = final_map.to_crs(epsg=4326)

# ============================================================
# 9. MERGE WITH GEO + EXPORT GeoJSON
# ============================================================
# ... [Keep your existing merge code] ...

final_map = final_map.to_crs(epsg=4326)

OUT_COLS = ['buurtcode', 'buurtnaam'] + SCEN_COLS + THEME_COLS + ['geometry']
export_df = final_map[[c for c in OUT_COLS if c in final_map.columns]]

# THE FIX: Add COORDINATE_PRECISION=5 to the export
export_df.to_file(
    'map_scenarios.geojson', 
    driver='GeoJSON',
    engine='pyogrio', # Usually faster, optional but recommended
    COORDINATE_PRECISION=5
)

print(f"\n✅  map_scenarios.geojson exported with reduced coordinate precision!")
print("    Themes: heat · populations · socioeconomic · mental_health · built_env · access")

Base dataset: 11554 neighborhoods
  health_data.csv: 11 indicators merged
  Heat_values_1.csv: RIVMClass merged for 11554 neighborhoods

--- True Model Accuracy (Test Set) ---
R²:  0.7043
MAE: 25.4551 points off

Final Model Coefficients:
  PET_gem                                       +4.3241
  p_ink_li                                      +3.4020
  p_elderly                                     +4.0470
  p_children                                    +2.9901
  p_bj_me10                                     +0.6089
  HeelVeelStressAfg4Weken_17                    +0.9300

Scenario averages (shared 0-100 scale):
  risk_2025_baseline: mean=35.1  max=96.0
  risk_2050_baseline: mean=37.1  max=98.0
  risk_2075_baseline: mean=39.1  max=100.0
  risk_2050_greening: mean=33.9  max=94.8

GeoPackage loaded: 14668 neighborhoods
  CBS variables added and cleaned: ['dist_gp', 'dist_hospital', 'dist_pharmacy', 'dist_park', 'dist_green', 'dist_forest', 'gas_use', 'elec_use', 'pct_pre2000', 'pct_low_edu',

In [ ]:
# === COMPREHENSIVE 5-THEME VULNERABILITY MODEL (FINAL) ===
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
# === COMPREHENSIVE 5-THEME VULNERABILITY MODEL (FINAL) ===
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pandas as pd
import geopandas as gpd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def pct_rank(series, invert=False):
    s = pd.to_numeric(series, errors='coerce')
    r = s.rank(pct=True, na_option='keep') * 100
    return (100 - r) if invert else r

def theme_score(df, col_defs):
    parts, used = [], []
    for (col, invert) in col_defs:
        if col in df.columns and df[col].notna().sum() > 50:
            parts.append(pct_rank(df[col], invert))
            used.append(("~" if invert else "+") + col)
    if not parts:
        return pd.Series([np.nan] * len(df), index=df.index)
    print(f"      ✓ {len(used)}: {', '.join(used[:4])}...")
    return pd.concat(parts, axis=1).mean(axis=1)

print("="*100)
print("🔥 HEAT VULNERABILITY — 5 COMPREHENSIVE THEMES | 47+ INDICATORS")
print("="*100)

# DATA LOADING
print("\n1. Loading and merging all data sources...")
df = pd.read_csv('Data/Neighborhood_Master_Data.csv')
df = df.dropna(subset=['GoedErvarenGezondheid_4', 'PET_gem', 'p_ink_li', 'p_elderly', 'p_children', 'p_bj_me10', 'HeelVeelStressAfg4Weken_17'])
df['_mk'] = df['MergeKey'].astype(str).str.strip().str.upper()
print(f"   ✓ Base: {len(df)} neighborhoods")

try:
    cbs_df = pd.read_excel('Data/kwb2024.xlsx', na_values=['.', ''])
    cbs_df['gwb_clean'] = cbs_df['gwb_code_10'].astype(str).str.strip().str.upper()
    CBS_SELECT = {'a_inw': 'pop_total', 'a_00_14': 'pop_00_14', 'a_65_oo': 'pop_65_plus', 'a_1p_hh': 'n_single_hh', 'bev_dich': 'pop_density', 'p_bj_me10': 'pct_pre2010', 'p_bj_mi10': 'pct_post2010', 'p_koopw': 'pct_owner_hh', 'p_huurw': 'pct_rental_hh', 'g_gas': 'avg_gas_use', 'g_ele': 'avg_elec_use', 'p_ink_li': 'pct_low_income', 'p_ink_hi': 'pct_high_income', 'p_arb_wn': 'pct_unemployed', 'p_hh_li': 'pct_hh_low_income', 'a_opl_hw': 'n_high_edu', 'g_afs_hp': 'dist_gp_avg', 'g_afs_gs': 'dist_hosp_avg', 'g_afs_kv': 'dist_pharmacy_avg', 'a_opp_ha': 'area_total_ha', 'a_lan_ha': 'area_land_ha', 'a_wat_ha': 'area_water_ha', 'pst_dekp': 'pct_tree_cover', 'a_wmo_t': 'n_wmo_support', 'a_bedv': 'n_care_receive'}
    cbs_subset = cbs_df[['gwb_clean'] + list(CBS_SELECT.keys())].rename(columns=CBS_SELECT).groupby('gwb_clean', as_index=False).first()
    df = df.merge(cbs_subset, left_on='_mk', right_on='gwb_clean', how='left')
    print(f"   ✓ CBS: {len([c for c in CBS_SELECT.values() if c in df.columns])}/{len(CBS_SELECT)} vars")
except Exception as e:
    print(f"   CBS skip: {e}")

try:
    health_raw = pd.read_csv('Data/health_data.csv', sep=';', low_memory=False)
    HEALTH_SELECT = ['Codering_3', 'HoogRisicoAngststoornisOfDepressie_14', 'AngstDepressiegevoelensAfg4Weken_16', 'HeelVeelStressAfg4Weken_17', 'SuicideGedachtenDeLaatste12Maanden_15', 'BrozeGezondheid_18', 'BrozeGezondheidFysiekDomein_20', 'BrozeGezondheidPsychologischDomein_19', 'BrozeGezondheidSociaalDomein_21', 'Eenzaam_25', 'SterkEenzaam_26', 'EmotioneelEenzaam_27', 'SociaalEenzaam_28', 'KanGoedVerkoelingVindenBinnen_59', 'KanGoedVerkoelingVindenBuiten_60']
    avail = [c for c in HEALTH_SELECT if c in health_raw.columns]
    h = health_raw[avail].copy()
    for c in avail:
        if c != 'Codering_3':
            h[c] = pd.to_numeric(h[c], errors='coerce')
    h_agg = h.groupby('Codering_3', as_index=False).mean(numeric_only=True)
    h_agg['Codering_3'] = h_agg['Codering_3'].astype(str).str.strip().str.upper()
    df = df.merge(h_agg, left_on='_mk', right_on='Codering_3', how='left')
    print(f"   ✓ Health: {len([c for c in HEALTH_SELECT[1:] if c in df.columns])} indicators")
except Exception as e:
    print(f"   Health skip: {e}")

# DERIVED INDICATORS
print("2. Engineering derived indicators...")
if 'pop_total' in df.columns and 'n_single_hh' in df.columns:
    df['pct_single_hh'] = (df['n_single_hh'] / df['pop_total']) * 100
if 'pop_65_plus' in df.columns and 'pop_total' in df.columns:
    df['pct_elderly'] = (df['pop_65_plus'] / df['pop_total']) * 100
if 'pop_00_14' in df.columns and 'pop_total' in df.columns:
    df['pct_children'] = (df['pop_00_14'] / df['pop_total']) * 100
if 'area_land_ha' in df.columns and 'area_total_ha' in df.columns:
    df['pct_green_area'] = (df['area_land_ha'] / df['area_total_ha']) * 100
if 'area_water_ha' in df.columns and 'area_total_ha' in df.columns:
    df['pct_water_area'] = (df['area_water_ha'] / df['area_total_ha']) * 100
if 'n_care_receive' in df.columns and 'pop_total' in df.columns:
    df['pct_care_recipients'] = (df['n_care_receive'] / df['pop_total']) * 100
if 'n_wmo_support' in df.columns and 'pop_total' in df.columns:
    df['pct_wmo_recipients'] = (df['n_wmo_support'] / df['pop_total']) * 100
print("   ✓ Done")

# THEMES
print("\n3. Computing 5 thematic vulnerability scores...\n")
print("  [1] HEAT EXPOSURE (climate, building, UHI, energy)")
df['theme_heat_exposure'] = theme_score(df, [('PET_gem', False), ('avg_gas_use', False), ('avg_elec_use', False), ('pct_pre2010', False), ('pop_density', False), ('pct_tree_cover', True), ('pct_green_area', True), ('pct_water_area', True)])

print("  [2] VULNERABLE POPULATIONS (age, frailty, isolation, care)")
df['theme_vulnerable_pop'] = theme_score(df, [('pct_elderly', False), ('pct_children', False), ('pct_single_hh', False), ('BrozeGezondheid_18', False), ('BrozeGezondheidFysiekDomein_20', False), ('BrozeGezondheidSociaalDomein_21', False), ('pct_care_recipients', False), ('pct_wmo_recipients', False)])

print("  [3] SOCIOECONOMIC DEPRIVATION (income, education, employment)")
df['theme_socioeconomic'] = theme_score(df, [('pct_low_income', False), ('pct_hh_low_income', False), ('pct_unemployed', False), ('pct_rental_hh', False), ('n_high_edu', True), ('dist_gp_avg', False), ('dist_hosp_avg', False)])

print("  [4] MENTAL HEALTH & ISOLATION (stress, loneliness, frailty)")
df['theme_mental_health'] = theme_score(df, [('HeelVeelStressAfg4Weken_17', False), ('HoogRisicoAngststoornisOfDepressie_14', False), ('AngstDepressiegevoelensAfg4Weken_16', False), ('Eenzaam_25', False), ('SterkEenzaam_26', False), ('BrozeGezondheidPsychologischDomein_19', False)])

print("  [5] BUILT ENVIRONMENT & ACCESS (cooling, green space, healthcare)")
df['theme_built_environment'] = theme_score(df, [('KanGoedVerkoelingVindenBinnen_59', True), ('KanGoedVerkoelingVindenBuiten_60', True), ('pct_tree_cover', True), ('pct_green_area', True), ('dist_gp_avg', False), ('dist_hosp_avg', False)])

# EXPORT
print("\n4. Exporting GeoJSON with all themes...")
geo_df = gpd.read_file('Data/GevoelstemperatuurBuurt_2022.gpkg')
geo_df['_bk'] = geo_df['buurtcode'].astype(str).str.strip().str.upper()
df['_mk2'] = df['MergeKey'].astype(str).str.strip().str.upper()
THEME_COLS = ['theme_heat_exposure', 'theme_vulnerable_pop', 'theme_socioeconomic', 'theme_mental_health', 'theme_built_environment']
final_map = geo_df.merge(df[['MergeKey'] + THEME_COLS + ['_mk2']], left_on='_bk', right_on='_mk2', how='left')
final_map = final_map.drop(columns=['_bk', '_mk2', 'MergeKey'], errors='ignore')
merged_count = final_map[THEME_COLS[0]].notna().sum()
print(f"   Merged: {merged_count} / {len(geo_df)} neighborhoods")

final_map = final_map.to_crs(epsg=4326)
OUT_COLS = ['buurtcode', 'buurtnaam', 'gemeentenaam'] + THEME_COLS + ['geometry']
final_map[[c for c in OUT_COLS if c in final_map.columns]].to_file('map_scenarios_themes.geojson', driver='GeoJSON')

print(f"\n{'='*100}")
print(f"✅ EXPORT: map_scenarios_themes.geojson | {merged_count} neighborhoods | 5 themes")
print(f"{'='*100}")
    return pd.concat(parts, axis=1).mean(axis=1)

print("="*100)
print("🔥 HEAT VULNERABILITY — 5 COMPREHENSIVE THEMES | 47+ INDICATORS")
print("="*100)

# DATA LOADING
print("\n1. Loading and merging all data sources...")
df = pd.read_csv('Data/Neighborhood_Master_Data.csv')
df = df.dropna(subset=['GoedErvarenGezondheid_4', 'PET_gem', 'p_ink_li', 'p_elderly', 'p_children', 'p_bj_me10', 'HeelVeelStressAfg4Weken_17'])
df['_mk'] = df['MergeKey'].astype(str).str.strip().str.upper()
print(f"   ✓ Base: {len(df)} neighborhoods")

try:
    cbs_df = pd.read_excel('Data/kwb2024.xlsx', na_values=['.', ''])
    cbs_df['gwb_clean'] = cbs_df['gwb_code_10'].astype(str).str.strip().str.upper()
    CBS_SELECT = {'a_inw': 'pop_total', 'a_00_14': 'pop_00_14', 'a_65_oo': 'pop_65_plus', 'a_1p_hh': 'n_single_hh', 'bev_dich': 'pop_density', 'p_bj_me10': 'pct_pre2010', 'p_bj_mi10': 'pct_post2010', 'p_koopw': 'pct_owner_hh', 'p_huurw': 'pct_rental_hh', 'g_gas': 'avg_gas_use', 'g_ele': 'avg_elec_use', 'p_ink_li': 'pct_low_income', 'p_ink_hi': 'pct_high_income', 'p_arb_wn': 'pct_unemployed', 'p_hh_li': 'pct_hh_low_income', 'a_opl_hw': 'n_high_edu', 'g_afs_hp': 'dist_gp_avg', 'g_afs_gs': 'dist_hosp_avg', 'g_afs_kv': 'dist_pharmacy_avg', 'a_opp_ha': 'area_total_ha', 'a_lan_ha': 'area_land_ha', 'a_wat_ha': 'area_water_ha', 'pst_dekp': 'pct_tree_cover', 'a_wmo_t': 'n_wmo_support', 'a_bedv': 'n_care_receive'}
    cbs_subset = cbs_df[['gwb_clean'] + list(CBS_SELECT.keys())].rename(columns=CBS_SELECT).groupby('gwb_clean', as_index=False).first()
    df = df.merge(cbs_subset, left_on='_mk', right_on='gwb_clean', how='left')
    print(f"   ✓ CBS: {len([c for c in CBS_SELECT.values() if c in df.columns])}/{len(CBS_SELECT)} vars")
except Exception as e:
    print(f"   CBS skip: {e}")

try:
    health_raw = pd.read_csv('Data/health_data.csv', sep=';', low_memory=False)
    HEALTH_SELECT = ['Codering_3', 'HoogRisicoAngststoornisOfDepressie_14', 'AngstDepressiegevoelensAfg4Weken_16', 'HeelVeelStressAfg4Weken_17', 'SuicideGedachtenDeLaatste12Maanden_15', 'BrozeGezondheid_18', 'BrozeGezondheidFysiekDomein_20', 'BrozeGezondheidPsychologischDomein_19', 'BrozeGezondheidSociaalDomein_21', 'Eenzaam_25', 'SterkEenzaam_26', 'EmotioneelEenzaam_27', 'SociaalEenzaam_28', 'KanGoedVerkoelingVindenBinnen_59', 'KanGoedVerkoelingVindenBuiten_60']
    avail = [c for c in HEALTH_SELECT if c in health_raw.columns]
    h = health_raw[avail].copy()
    for c in avail:
        if c != 'Codering_3':
            h[c] = pd.to_numeric(h[c], errors='coerce')
    h_agg = h.groupby('Codering_3', as_index=False).mean(numeric_only=True)
    h_agg['Codering_3'] = h_agg['Codering_3'].astype(str).str.strip().str.upper()
    df = df.merge(h_agg, left_on='_mk', right_on='Codering_3', how='left')
    print(f"   ✓ Health: {len([c for c in HEALTH_SELECT[1:] if c in df.columns])} indicators")
except Exception as e:
    print(f"   Health skip: {e}")

# DERIVED INDICATORS
print("2. Engineering derived indicators...")
if 'pop_total' in df.columns and 'n_single_hh' in df.columns:
    df['pct_single_hh'] = (df['n_single_hh'] / df['pop_total']) * 100
if 'pop_65_plus' in df.columns and 'pop_total' in df.columns:
    df['pct_elderly'] = (df['pop_65_plus'] / df['pop_total']) * 100
if 'pop_00_14' in df.columns and 'pop_total' in df.columns:
    df['pct_children'] = (df['pop_00_14'] / df['pop_total']) * 100
if 'area_land_ha' in df.columns and 'area_total_ha' in df.columns:
    df['pct_green_area'] = (df['area_land_ha'] / df['area_total_ha']) * 100
if 'area_water_ha' in df.columns and 'area_total_ha' in df.columns:
    df['pct_water_area'] = (df['area_water_ha'] / df['area_total_ha']) * 100
if 'n_care_receive' in df.columns and 'pop_total' in df.columns:
    df['pct_care_recipients'] = (df['n_care_receive'] / df['pop_total']) * 100
if 'n_wmo_support' in df.columns and 'pop_total' in df.columns:
    df['pct_wmo_recipients'] = (df['n_wmo_support'] / df['pop_total']) * 100
print("   ✓ Done")

# THEMES
print("\n3. Computing 5 thematic vulnerability scores...\n")
print("  [1] HEAT EXPOSURE (climate, building, UHI, energy)")
df['theme_heat_exposure'] = theme_score(df, [('PET_gem', False), ('avg_gas_use', False), ('avg_elec_use', False), ('pct_pre2010', False), ('pop_density', False), ('pct_tree_cover', True), ('pct_green_area', True), ('pct_water_area', True)])

print("  [2] VULNERABLE POPULATIONS (age, frailty, isolation, care)")
df['theme_vulnerable_pop'] = theme_score(df, [('pct_elderly', False), ('pct_children', False), ('pct_single_hh', False), ('BrozeGezondheid_18', False), ('BrozeGezondheidFysiekDomein_20', False), ('BrozeGezondheidSociaalDomein_21', False), ('pct_care_recipients', False), ('pct_wmo_recipients', False)])

print("  [3] SOCIOECONOMIC DEPRIVATION (income, education, employment)")
df['theme_socioeconomic'] = theme_score(df, [('pct_low_income', False), ('pct_hh_low_income', False), ('pct_unemployed', False), ('pct_rental_hh', False), ('n_high_edu', True), ('dist_gp_avg', False), ('dist_hosp_avg', False)])

print("  [4] MENTAL HEALTH & ISOLATION (stress, loneliness, frailty)")
df['theme_mental_health'] = theme_score(df, [('HeelVeelStressAfg4Weken_17', False), ('HoogRisicoAngststoornisOfDepressie_14', False), ('AngstDepressiegevoelensAfg4Weken_16', False), ('Eenzaam_25', False), ('SterkEenzaam_26', False), ('BrozeGezondheidPsychologischDomein_19', False)])

print("  [5] BUILT ENVIRONMENT & ACCESS (cooling, green space, healthcare)")
df['theme_built_environment'] = theme_score(df, [('KanGoedVerkoelingVindenBinnen_59', True), ('KanGoedVerkoelingVindenBuiten_60', True), ('pct_tree_cover', True), ('pct_green_area', True), ('dist_gp_avg', False), ('dist_hosp_avg', False)])

# EXPORT
print("\n4. Exporting GeoJSON with all themes...")
geo_df = gpd.read_file('Data/GevoelstemperatuurBuurt_2022.gpkg')
geo_df['_bk'] = geo_df['buurtcode'].astype(str).str.strip().str.upper()
df['_mk2'] = df['MergeKey'].astype(str).str.strip().str.upper()
THEME_COLS = ['theme_heat_exposure', 'theme_vulnerable_pop', 'theme_socioeconomic', 'theme_mental_health', 'theme_built_environment']
final_map = geo_df.merge(df[['MergeKey'] + THEME_COLS + ['_mk2']], left_on='_bk', right_on='_mk2', how='left')
final_map = final_map.drop(columns=['_bk', '_mk2', 'MergeKey'], errors='ignore')
merged_count = final_map[THEME_COLS[0]].notna().sum()
print(f"   Merged: {merged_count} / {len(geo_df)} neighborhoods")

final_map = final_map.to_crs(epsg=4326)
OUT_COLS = ['buurtcode', 'buurtnaam', 'gemeentenaam'] + THEME_COLS + ['geometry']
final_map[[c for c in OUT_COLS if c in final_map.columns]].to_file('map_scenarios_themes.geojson', driver='GeoJSON')

print(f"\n{'='*100}")
print(f"✅ EXPORT: map_scenarios_themes.geojson | {merged_count} neighborhoods | 5 themes")
print(f"{'='*100}")